In [ ]:
import pandas as pd

data = pd.read_csv('./data/daF3077_eng.csv', sep=';')

## do you need to do cleaning of the data?

## Unsupervised learning

Unsupervised learning focuses on working without a target variable.
The goal is to find patterns in the data, such as clusters of similar observations or associations between variables.
This means there is no _correct_ outcome like in supervised machine learning, but rather various different outcomes.

We examine how to cluster different welfare needs (`v1_x`) to summarise what kind of groups emerge in the needs.

In [ ]:
variables = ['v1_1', 'v1_2', 'v1_3', 'v1_4', 'v1_5', 'v1_6']

In [ ]:
to_cluster = data[variables]
print(to_cluster.describe())

## Clustering

Clustering groups together similar observations from the data.

### $k$-means

$k$-means method identifies _centroids_ that summarise the response patterns of participants within each cluster (and assigns each responder into one of the clusters).
[Documentation](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html)

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=3)
kmeans.fit(to_cluster)

cluster_labels = kmeans.labels_
print(pd.Series(cluster_labels).value_counts())

### Hierarchical clustering

Hierarchical clustering operates by separating the data into branches based on similarity.
[Documentation](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.AgglomerativeClustering.html)

In [ ]:
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import dendrogram, linkage

linkage_matrix = linkage(to_cluster, method='ward')

In [ ]:
plt.figure(figsize=(10, 6))
dendrogram(linkage_matrix, truncate_mode='lastp', p=20)
plt.title('Cluster Dendrogram')
plt.xlabel('Sample index or (cluster size)')
plt.ylabel('Distance')
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.cluster import AgglomerativeClustering

hierarchical = AgglomerativeClustering(n_clusters=3, linkage='ward')
hclust_labels = hierarchical.fit_predict(to_cluster)

print(pd.Series(hclust_labels).value_counts().sort_index())

In [ ]:
## DBSCAN

In [ ]:
from sklearn.cluster import DBSCAN

dbscan = DBSCAN(eps=1, min_samples=50)
dbscan_labels = dbscan.fit_predict(to_cluster)

print(pd.Series(dbscan_labels).value_counts())

## Tasks

* Check what these different clustering algorithms produce and describe the clusters.
* Check similarities and differences between clustering outcomes across methods.
* Vary the number of clusters and seeds to examine the stability of the outcomes.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaled = scaler.fit_transform(to_cluster)
scaled_df = pd.DataFrame(scaled, columns=variables)

print(scaled_df.head())

## Tasks

* Try out variables which were not all on the same scale.
How does the result differ on scaled and non-scaled clusters?

# Dimension reduction

### Principal component analysis

Reduces many correlated variables into a smaller set of components capturing the most variance in the data.
[Documentation](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaled = scaler.fit_transform(to_cluster)

In [ ]:
from sklearn.decomposition import PCA

# scale=True in R's prcomp corresponds to StandardScaler before PCA
pca = PCA()
pca_result = pca.fit_transform(scaled)

In [ ]:
import numpy as np

loadings = pd.DataFrame(
    pca.components_.T,
    index=variables,
    columns=[f'PC{i+1}' for i in range(pca.n_components_)]
)
print(loadings)

fig, ax = plt.subplots(figsize=(8, 6))

ax.scatter(pca_result[:, 0], pca_result[:, 2], alpha=0.3, s=10, label='Observations')

# Plot the loadings (variable arrows)
scale_factor = 3  # scale arrows for visibility
for i, var in enumerate(variables):
    ax.annotate(
        '', 
        xy=(loadings.iloc[i, 0] * scale_factor, loadings.iloc[i, 2] * scale_factor),
        xytext=(0, 0),
        arrowprops=dict(arrowstyle='->', color='red', lw=1.5)
    )
    ax.text(
        loadings.iloc[i, 0] * scale_factor * 1.1,
        loadings.iloc[i, 2] * scale_factor * 1.1,
        var, color='red', fontsize=10
    )

ax.axhline(0, color='grey', linestyle='--', linewidth=0.5)
ax.axvline(0, color='grey', linestyle='--', linewidth=0.5)
ax.set_xlabel('PC1')
ax.set_ylabel('PC3')
ax.set_title('Biplot (PC1 vs PC3)')
plt.tight_layout()
plt.show()

### UMAP

A nonlinear method that maps high-dimensional data into a low-dimensional space.
Use `n_components` to change the number of components coming out. [Documentation](https://umap-learn.readthedocs.io/en/latest/)

In [ ]:
import umap

In [ ]:
reducer = umap.UMAP(n_components=5)
umap_result = reducer.fit_transform(scaled)

In [ ]:
umap_df = pd.DataFrame(umap_result)
print(umap_df.head())

plt.figure(figsize=(8, 6))
plt.scatter(umap_result[:, 0], umap_result[:, 2], alpha=0.3, s=10)
plt.xlabel('UMAP Component 1')
plt.ylabel('UMAP Component 3')
plt.title('UMAP: Component 1 vs Component 3')
plt.tight_layout()
plt.show()

## Tasks

* Add to above visualisations different genders and ages from the data.
* Add to above visualisations clusters from above.
* Try to interpret different components?
* Can you find a set of variables where components are more interpretable? 
* Try using PCA results in a regression model as predictors. Do they perform better than original variables?